# Deliverable 2 - INF201 

**Forfattere:** Amena og Arthika

**GitHub-repository:** https://github.com/amhus8955/INF201_deliverable_2

I denne notebooken lager vi en pipeline som finner alle loggfiler fra kassene, leser dem (blandede tegnkodinger), fjerner systemstøy, renser prisfeil og lager fire rapportfiler. Task 1-3 definerer funksjonene for innlesing, rekursive søk og NumPy-rensing. Task 4 analyserer data med Pandas, og Task 5 setter alt sammen til én arbeidsflyt.

## Scenario  - Campus Coffee Shop Sales Pipeline
You have been hired as a Python developer for the Campus Coffee Shop Chain. The chain operates several coffee stands across different campus buildings. This is an international campus, so the stands accept payment in several currencies.

Each cash register saves transaction logs as text files. However, the legacy system has several issues:
* Mixed Character Encodings: Older registers write text files using ISO-8859-1 (Latin-1) encoding, while newer registers write files using UTF-8.
* System Noise: Registers intersperse valid sales lines with printer and network debug log lines.
* Nested Folder Hierarchy: Log files are stored across nested subdirectories organized by year, store, and register ID.
* Data Errors: System glitches record occasional extreme price outliers (e.g., $999.00), and prices in NOK are by error registered using EUR-scale values (e.g., 4.50 NOK instead of 45.00 NOK).

Your task is to set up Python code that discovers all log files, cleans the transaction data, calculates price metrics and generates summary reports.<br>
Note: In this assignment, you write the functions you will use in tasks 1-4, and then put everything together in task 5. You should therefore read through the entire assignment text before you start, to fully understand what you will use each function for.

### Overview of inputs and outputs for this assignment
Inputs (in `data/` folder)
* You generate the input data on you own laptop by running the script `generate_data_for_d2.py`
* `data/store_config.json`: A nested JSON file containing store metadata.
* `data/2026/...`: A folder hierarchy containing subdirectories with .log files in mixed character encodings.

Outputs (files your code should generate)
* `cleaned_transactions.csv`: A CSV table containing all parsed, outlier-cleaned, and currency-corrected transactions.
* `hourly_revenue.json`: A JSON report containing total revenue per hour per item.
* `status_pivot_table.csv`: A CSV file containing a pivot table of total transaction revenue grouped by item across status categories.
* `item_refund_ratios.csv`: A CSV table containing refund ratios for each item type.

## Task 1: File ingestion and regular expressions

1. Write a robust file reader function:
    * Use `pathlib.Path` to handle file paths.
    * The function should be able to handle both `UTF-8` and `ISO-8859-1/Latin-1` encoding. You may assume that these are the only two encodings the files can have. Note that the function should handle the encoding, you may not hard-code which encoding each file has.
    * Return a tuple containing two strings: `(file_text_content, encoding_used)`.
2. Write a log parser function that uses regular expressions.
    Log files contain valid sales lines mixed with system noise.<br>
    Valid line format:<br>
    `[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)`<br>
    Noise lines to ignore:<br>
    `--- PRINTER PAPER LOW WARNING ---`<br>
    `=== NETWORK SYNC OK ===`<br>
    * Construct a regular expression (`re` module) to capture the 6 key fields from valid lines:
        * `timestamp` (string, format: `YYYY-MM-DD HH:MM:SS`)
        * `register_id` (string, e.g., `POS_01`)
        * `item` (string, e.g., `Latte`)
        * `price` (float, e.g., `4.50`)
        * `currency` (string, e.g., `EUR`, `NOK`, `USD`)
        * `status` (string: `SUCCESS`, `REFUND`, or `CANCELLED`)
    * Return a list of Python dictionaries, where each dictionary represents one valid transaction record.

In [1]:
# Deliverable 2: Campus Coffee Shop Sales Pipeline
# Task 1: File ingestion and regular expressions
# Task 2: Recursive utility functions
# Task 3: Functions for cleaning and analysing data using NumPy
 
# Makes type hints like str | Path work on Python 3.9.
from __future__ import annotations
 
import json
import re
from pathlib import Path
from typing import Any
 
import numpy as np
import numpy.typing as npt
 
DATA_DIR = Path('data')
# Epsilon from the assignment that prevents division by zero.
EPSILON = 1e-6
 
 
# I used named groups (?P<name>...) so that each field gets a name.
# Then match.groupdict() returns a ready-made dictionary instead
# of getting the fields one by one with number indexes.
LOG_LINE_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>[A-Za-z0-9_]+)\]\s+"
    # Non-greedy (.+?) so item names with spaces also work.
    r"item=(?P<item>.+?)\s+"
    r"price=(?P<price>\d+(?:\.\d+)?)\s+"
    r"(?P<currency>[A-Z]{3})\s+"
    # Only the three valid statuses are accepted.
    # $ = end of the line, so lines with extra text after
    # the status are not counted as valid.
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$"
)

In [2]:
# %%
# Task 1: File ingestion and regular expressions
 
 
# Robust file reader
def read_log_file(file_path: str | Path) -> tuple[str, str]:
    """
    Read a text file that is encoded in either UTF-8 or ISO-8859-1.
 
    The file is read as raw bytes and decoded as UTF-8 first. If that
    fails it's decoded as ISO-8859-1 (Latin-1), so the encoding does
    not have to be hard-coded for each file.
 
    Parameters
    ----------
    file_path : str | Path
        Path to the file that should be read.
 
    Returns
    -------
    tuple[str, str]
        The text content of the file and the name of the encoding used.
 
    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    """
    # Reading bytes lets the same content be decoded more than once.
    raw_bytes = Path(file_path).read_bytes()
    try:
        return raw_bytes.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        # Latin-1 can decode any byte, so it must be tried last. If it
        # came first, UTF-8 files would be read with wrong characters.
        return raw_bytes.decode("iso-8859-1"), "iso-8859-1"
 
 
# Log parser using regular expressions
def parse_log_text(log_text: str) -> list[dict[str, str | float]]:
    """
    Extract valid transaction records from the text of a log file.
 
    Lines that do not match the sales format, such as printer warnings
    and network messages, are ignored.
 
    Parameters
    ----------
    log_text : str
        The full text content of a log file.
 
    Returns
    -------
    list[dict[str, str | float]]
        One dictionary per valid transaction with the keys
        ``timestamp``, ``register_id``, ``item``, ``price``,
        ``currency`` and ``status``. ``price`` is a float, the other
        values are strings.
    """
    records: list[dict[str, str | float]] = []
    # splitlines() handles both \n (Mac/Linux) and \r\n (Windows),
    # so line breaks from different cash registers are treated equally.
    for line in log_text.splitlines():
        match = LOG_LINE_PATTERN.match(line.strip())
 
        # Noise lines are skipped because they do not match.
        if match is None:
            continue
        record: dict[str, str | float] = match.groupdict()
 
        # The price is converted to a float because the assignment
        # requires it, and because it is used in NumPy and pandas later.
        record["price"] = float(match.group("price"))
        records.append(record)
 
    return records
 
 
# Test of Task 1
# A small test string with both valid lines and noise, to check that
# the parser only extracts the sales lines.
example_text = (
    "--- PRINTER PAPER LOW ---\n"
    "[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)\n"
    "=== NETWORK SYNC OK ===\n"
    "[DEBUG] BARCODE SCANNER REBOOTED\n"
    "[2026-10-15 08:32:10] [POS_01] item=Muffin price=999.00 NOK (REFUND)\n"
)
 
print("--- Task 1: File ingestion and regular expressions ---")
parsed = parse_log_text(example_text)
for transaction in parsed:
    print(transaction)
 
# Only the two sales lines should be kept, not the three noise lines.
assert len(parsed) == 2
assert parsed[0]["item"] == "Latte"
assert isinstance(parsed[0]["price"], float)
assert parsed[1]["status"] == "REFUND"

--- Task 1: File ingestion and regular expressions ---
{'timestamp': '2026-10-15 08:30:00', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.5, 'currency': 'EUR', 'status': 'SUCCESS'}
{'timestamp': '2026-10-15 08:32:10', 'register_id': 'POS_01', 'item': 'Muffin', 'price': 999.0, 'currency': 'NOK', 'status': 'REFUND'}


### Diskusjon: Task 1

- `read_log_file` leser filen som bytes og prøver UTF-8 først. Hvis det feiler, brukes ISO-8859-1. Latin-1 må komme sist fordi den kan dekode alle bytes.
- `parse_log_text` bruker et regulært uttrykk med navngitte grupper. Uttrykket må matche hele linjen, så støylinjer hoppes over.
- Prisen gjøres om til `float`.

## Task 2: Recursive utility functions

1. Write a recursive directory search function that finds all log files in a nested directory and returns a list of `pathlib.Path` objects:
    * Convert the input path to a `pathlib.Path` object.
    * Iterate through items using `Path.iterdir()`.
    * If an item is a directory, recursively search inside it.
    * If an item is a file with a matching extension (`.log`), add its Path object to a list.
    * Constraint: Do NOT use `Path.glob()`, `Path.rglob()`, or `os.walk()`. You must implement explicit recursion.
2. Metadata about the different stores can be found in the `data/store_config.json` file. This file contains deeply nested dictionaries and lists (see example below). Write a recursive function that:
    * Traverses any combination of nested dictionaries and lists found in this file.
    * Returns a set (type hint: `set[str]`) of all register id strings where `"active": true`.

Example config.json:
```python
{
  "company": "Campus Coffee",
  "regions": [
    {
      "region_name": "Main Campus",
      "stores": [
        {
          "name": "Library Cafe",
          "registers": [
            {"id": "POS_01", "active": true},
            {"id": "POS_02", "active": false}
          ]
        }
      ]
    }
  ]
}
```

In [3]:
# %%
# Task 2: Recursive utility functions
 
 
# Recursive directory search
def find_log_files(directory: str | Path,
                   extension: str = ".log") -> list[Path]:
    """
    Recursively find all files with a given extension in a directory.
 
    The function goes through every item in the directory with
    ``Path.iterdir()``. Subdirectories are searched by calling the
    function again (recursion). ``glob``, ``rglob`` and ``os.walk`` are
    deliberately not used.
 
    Parameters
    ----------
    directory : str | Path
        The directory to start searching in.
    extension : str, optional
        File extension to look for, by default ".log".
 
    Returns
    -------
    list[Path]
        Paths to all matching files in the directory and all its
        subdirectories, sorted alphabetically within each folder.
 
    Raises
    ------
    FileNotFoundError
        If the directory does not exist.
    """
    # The assignment requires the input to be converted to a Path.
    # This lets the function accept both "data" (str) and Path("data").
    directory = Path(directory)
    found_files: list[Path] = []
 
    # sorted() gives the same order on every computer.
    for item in sorted(directory.iterdir()):
        if item.is_dir():
            # The recursion stops when a folder has no subfolders.
            found_files.extend(find_log_files(item, extension))
        elif item.is_file() and item.suffix == extension:
            found_files.append(item)
 
    return found_files
 
 
# Helper function for reading the config file
def load_store_config(config_path: str | Path) -> dict[str, Any]:
    """
    Load the store configuration from a JSON file.
 
    Parameters
    ----------
    config_path : str | Path
        Path to ``store_config.json``.
 
    Returns
    -------
    dict[str, Any]
        The JSON content as nested Python dictionaries and lists.
    """
    with Path(config_path).open(encoding="utf-8") as file:
        return json.load(file)
 
 
# Recursive search for active register IDs
def find_active_registers(data: Any) -> set[str]:
    """
    Recursively collect the IDs of all active registers.
 
    Traverses any combination of nested dictionaries and lists. A
    dictionary counts as an active register if it has an ``"id"`` key
    with a string value and ``"active"`` set to ``True``.
 
    Parameters
    ----------
    data : Any
        Nested structure of dictionaries and lists, for example the
        content of ``store_config.json``.
 
    Returns
    -------
    set[str]
        The IDs of all registers where ``"active"`` is ``True``.
    """
    # I use a set because the assignment requires it, and because a set
    # removes duplicates if the same ID appears in multiple places.
    active_ids: set[str] = set()
 
    if isinstance(data, dict):
        # "is True" so that e.g. the number 1 does not count as active.
        register_id = data.get("id")
        if data.get("active") is True and isinstance(register_id, str):
            active_ids.add(register_id)
 
        # Lists and dictionaries can be nested inside the values.
        for value in data.values():
            # |= combines two sets (union).
            active_ids |= find_active_registers(value)
 
    elif isinstance(data, list):
        for element in data:
            active_ids |= find_active_registers(element)
 
    # Strings, numbers and booleans have no deeper levels, so the
    # recursion stops here (base case).
    return active_ids
 
 
# Test of Task 2
print("\n--- Task 2: Recursive utility functions ---")
 
# The example from the assignment, where only POS_01 is active.
example_config = {
    "company": "Campus Coffee",
    "regions": [
        {
            "region_name": "Main Campus",
            "stores": [
                {
                    "name": "Library Cafe",
                    "registers": [
                        {"id": "POS_01", "active": True},
                        {"id": "POS_02", "active": False},
                    ],
                }
            ],
        }
    ],
}
assert find_active_registers(example_config) == {"POS_01"}
 
# The real data from generate_data_for_d2.py. The data folder must be
# in the same folder as this file.
log_files = find_log_files(DATA_DIR)
print(f"Found {len(log_files)} log files:")
for log_file in log_files:
    print(" ", log_file)
 
store_config = load_store_config(DATA_DIR / "store_config.json")
active_registers = find_active_registers(store_config)
print("Active registers:", sorted(active_registers))
 
# 3 stores x 3 registers = 9 log files.
assert len(log_files) == 9
assert active_registers == {"POS_01", "POS_02", "POS_03"}


--- Task 2: Recursive utility functions ---
Found 9 log files:
  data/2026/library_cafe/POS_01/register_sales.log
  data/2026/library_cafe/POS_02/register_sales.log
  data/2026/library_cafe/POS_03/register_sales.log
  data/2026/science_building/POS_01/register_sales.log
  data/2026/science_building/POS_02/register_sales.log
  data/2026/science_building/POS_03/register_sales.log
  data/2026/student_union/POS_01/register_sales.log
  data/2026/student_union/POS_02/register_sales.log
  data/2026/student_union/POS_03/register_sales.log
Active registers: ['POS_01', 'POS_02', 'POS_03']


### Diskusjon: Task 2

- `find_log_files` bruker rekursjon med `Path.iterdir()`. For hver undermappe kaller funksjonen seg selv, og filer med endelsen `.log` legges i listen.
- `find_active_registers` går rekursivt gjennom nøstede dictionaries og lister og samler id-en til alle kasser med `"active": true` i et `set`.
- Funksjonene ble testet med eksempelet fra oppgaven og på de ekte dataene (9 loggfiler, aktive kasser `POS_01`-`POS_03`).

## Task 3: Functions for cleaning and analysing data using NumPy
1. Write a function for removing price outliers that:
    * Converts input price data to a 1D NumPy float array.
    * Detects hardware glitch outliers ($999.00$) where a value deviates from the mean by more than 3 standard deviations:$$\lvert x - \mu \rvert > 3\sigma$$
    * Replaces detected outlier values with the median of the valid prices.
    * Returns the cleaned 1D array.
2. Write a function `compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray` that:
    * Computes the element-wise refund ratio using NumPy array operations:
    $$\text{Refund Ratio} = \frac{\text{Refund Count}}{\text{Sales Count} + 10^{-6}}$$
    * The addition of $\epsilon = 10^{-6}$ prevents zero-division errors.

In [4]:
# %%
# Task 3: Functions for cleaning and analysing data using NumPy
 
 
# Remove price outliers
def remove_price_outliers(prices: npt.ArrayLike,
                          n_std: float = 3.0) -> np.ndarray:
    """
    Replace extreme price outliers with the median of the valid prices.
 
    A value is treated as an outlier if it deviates from the mean by
    more than ``n_std`` standard deviations, i.e.
    ``abs(x - mean) > n_std * std``.
 
    Parameters
    ----------
    prices : npt.ArrayLike
        Price data, e.g. a list, a pandas Series or a NumPy array.
    n_std : float, optional
        Number of standard deviations used as the limit, by default 3.0.
 
    Returns
    -------
    np.ndarray
        A cleaned 1D float array of the same length as the input.
    """
    # np.array makes a copy, so the original data is not changed.
    cleaned = np.array(prices, dtype=float).ravel()
 
    # mean() of an empty array gives a warning, so return early.
    if cleaned.size == 0:
        return cleaned
 
    mean = cleaned.mean()
    std = cleaned.std()
 
    # True where the value is an outlier. Calculated for the whole array
    # at once (vectorised) instead of with a for loop.
    is_outlier = np.abs(cleaned - mean) > n_std * std
 
    # The median uses only the valid prices, so the 999 values do not
    # pull it up. The median is less affected by extremes than the mean.
    if is_outlier.any():
        valid_median = np.median(cleaned[~is_outlier])
        cleaned[is_outlier] = valid_median
 
    return cleaned
 
 
# Compute refund ratio
def compute_refund_ratio(sales_count: np.ndarray,
                         refund_count: np.ndarray) -> np.ndarray:
    """
    Compute the element-wise refund ratio.
 
    Uses ``refund_count / (sales_count + 1e-6)``. The small epsilon
    prevents division by zero when an item has no sales.
 
    Parameters
    ----------
    sales_count : np.ndarray
        Number of sales per item.
    refund_count : np.ndarray
        Number of refunds per item, in the same order as
        ``sales_count``.
 
    Returns
    -------
    np.ndarray
        Refund ratio for each element.
    """
    # float, so lists and pandas Series also work as input.
    sales = np.asarray(sales_count, dtype=float)
    refunds = np.asarray(refund_count, dtype=float)
 
    # NumPy divides element by element.
    return refunds / (sales + EPSILON)
 
 
# Test of Task 3
print("\n--- Task 3: Cleaning and analysing data using NumPy ---")
 
# The 3-sigma rule needs enough data points. With only a few prices, one
# outlier pulls up both the mean and the std so much that it is missed.
test_prices = [3.0, 4.5, 4.2, 3.5, 3.8, 7.5, 3.1, 4.4, 4.0, 3.6,
               3.9, 7.2, 2.9, 4.6, 4.3, 3.4, 3.7, 7.8, 3.2, 4.1, 999.0]
cleaned_test = remove_price_outliers(test_prices)
print("Price before:", test_prices[-1], "-> after:", cleaned_test[-1])
assert 999.0 not in cleaned_test
assert len(cleaned_test) == len(test_prices)
 
# All the real prices from the log files (uses Task 1 and 2).
all_records = []
for log_file in log_files:
    text, _ = read_log_file(log_file)
    all_records.extend(parse_log_text(text))
 
real_prices = np.array([record["price"] for record in all_records])
cleaned_real = remove_price_outliers(real_prices)
print("Number of prices:", real_prices.size)
print("999 glitches before cleaning:", int(np.sum(real_prices == 999.0)))
print("999 glitches after cleaning:", int(np.sum(cleaned_real == 999.0)))
assert not np.any(cleaned_real == 999.0)
 
# The last item has no sales, to check that there is no zero division.
ratios = compute_refund_ratio(np.array([10, 4, 0]), np.array([2, 1, 0]))
print("Refund ratios:", np.round(ratios, 3))
assert np.allclose(ratios, [0.2, 0.25, 0.0])


--- Task 3: Cleaning and analysing data using NumPy ---
Price before: 999.0 -> after: 3.95
Number of prices: 720
999 glitches before cleaning: 19
999 glitches after cleaning: 0
Refund ratios: [0.2  0.25 0.  ]


### Diskusjon: Task 3

- `remove_price_outliers` regner ut gjennomsnitt og standardavvik, og erstatter verdier med |x − gjennomsnitt| > 3 standardavvik med medianen av de gyldige prisene. Medianen brukes fordi den ikke påvirkes av de ekstreme verdiene.
- `compute_refund_ratio` regner ut `refund_count / (sales_count + 1e-6)` for hvert element. Epsilon hindrer deling på null.
- Funksjonene ble testet på en liste med én 999.00-verdi, på alle de ekte prisene, og på et tilfelle med null salg.

## Task 4: Data analysis using Pandas
1. Write a function that:
    * Converts the list of transaction dictionaries read from the log files into a `pandas.DataFrame`.
    * Converts the `timestamp` column to `datetime` and sets it as the DataFrame index.
    * Sorts the DataFrame chronologically by timestamp.
2. Write a function that has the DataFrame you just created as input, groups data by item and resamples transactions in 1-hour windows (1h), calculating total price (sum).
3. Write a function that has the DataFrame as input  and creates a pivot table displaying total transaction revenue grouped by item across status categories (`SUCCESS`, `REFUND`, `CANCELLED`).
4. Write a function that has the DataFrame as input and calculates the refund ratio per item by calling the function you made in task 3.

In [5]:
import pandas as pd

STATUSES = ["SUCCESS", "REFUND", "CANCELLED"]


def build_dataframe(records: list[dict]) -> pd.DataFrame:
    """Lager DataFrame, setter timestamp som indeks og sorterer."""
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df.set_index("timestamp").sort_index()


def hourly_revenue(df: pd.DataFrame) -> pd.Series:
    """Sum av pris per item per time."""
    return df.groupby("item")["price"].resample("1h").sum()


def status_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """Omsetning per item (rader) og status (kolonner)."""
    pivot = df.pivot_table(index="item", columns="status", values="price",
                           aggfunc="sum", fill_value=0.0)
    return pivot.reindex(columns=STATUSES, fill_value=0.0)


def item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Refund ratio per item. Bruker compute_refund_ratio fra Task 3."""
    counts = (df.groupby(["item", "status"]).size()
                .unstack(fill_value=0)
                .reindex(columns=STATUSES, fill_value=0))
    result = pd.DataFrame({"sales_count": counts["SUCCESS"],
                           "refund_count": counts["REFUND"]})
    result["refund_ratio"] = compute_refund_ratio(
        result["sales_count"].to_numpy(), result["refund_count"].to_numpy())
    return result

In [6]:
test = [
    {"timestamp": "2026-10-15 08:10:00", "register_id": "POS_01", "item": "Latte",
     "price": 4.5, "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 07:50:00", "register_id": "POS_01", "item": "Latte",
     "price": 4.4, "currency": "EUR", "status": "REFUND"},
]
t = build_dataframe(test)
print(t)
print(hourly_revenue(t))
print(status_pivot_table(t))
print(item_refund_ratios(t))

                    register_id   item  price currency   status
timestamp                                                      
2026-10-15 07:50:00      POS_01  Latte    4.4      EUR   REFUND
2026-10-15 08:10:00      POS_01  Latte    4.5      EUR  SUCCESS
item   timestamp          
Latte  2026-10-15 07:00:00    4.4
       2026-10-15 08:00:00    4.5
Name: price, dtype: float64
status  SUCCESS  REFUND  CANCELLED
item                              
Latte       4.5     4.4        0.0
       sales_count  refund_count  refund_ratio
item                                          
Latte            1             1      0.999999


### Diskusjon: Task 4

- `build_dataframe` lager en DataFrame, gjør `timestamp` om til datetime, bruker den som indeks og sorterer kronologisk.
- `hourly_revenue` grupperer på `item` og summerer prisen i 1-timesvinduer.
- `status_pivot_table` viser omsetning per item og status.
- `item_refund_ratios` teller `SUCCESS` som salg og `REFUND` som refusjoner, og bruker `compute_refund_ratio` fra Task 3.

## Task 5: Create full workflow
Combine the functions you have made into a workflow that does the following:
1. Recursively find all log files and extract active register IDs.
2. Ingest text logs and parse records into a list of dictionaries.
3. Build the initial Pandas DataFrame.
4. Apply NumPy outlier cleaning to remove $999.00$ price glitches.
5. Apply currency correction: multiply price by $10.0$ for all transactions where `currency == "NOK"`.
6. Generate resampled reports, pivot tables and refund ratios.
7. Save all output files listed in the introduction (Scenario: Overview of inputs and outputs for this assignment).

In [7]:
OUTPUT_DIR = Path("output")


def run_pipeline():
    OUTPUT_DIR.mkdir(exist_ok=True)

    # 1. Finn loggfiler og aktive kasser
    log_files = find_log_files(DATA_DIR)
    active_ids = find_active_registers(load_store_config(DATA_DIR / "store_config.json"))

    # 2. Les og parse alle filer
    records = []
    for path in log_files:
        text, encoding = read_log_file(path)
        records.extend(parse_log_text(text))
    records = [r for r in records if r["register_id"] in active_ids]

    # 3. DataFrame
    df = build_dataframe(records)

    # 4. Fjern outliers (999.00)
    df["price"] = remove_price_outliers(df["price"].to_numpy())

    # 5. NOK-korreksjon: gang med 10
    is_nok = df["currency"] == "NOK"
    df.loc[is_nok, "price"] = df.loc[is_nok, "price"] * 10.0

    # 6. Rapporter
    hourly = hourly_revenue(df)
    pivot = status_pivot_table(df)
    ratios = item_refund_ratios(df)

    # 7. Lagre de fire filene
    df.round(2).to_csv(OUTPUT_DIR / "cleaned_transactions.csv")

    hourly_dict = {}
    for (item, hour), value in hourly.items():
        hourly_dict.setdefault(item, {})[hour.strftime("%Y-%m-%d %H:%M:%S")] = round(float(value), 2)
    with open(OUTPUT_DIR / "hourly_revenue.json", "w", encoding="utf-8") as f:
        json.dump(hourly_dict, f, indent=2)

    pivot.round(2).to_csv(OUTPUT_DIR / "status_pivot_table.csv")
    ratios.round(4).to_csv(OUTPUT_DIR / "item_refund_ratios.csv")

    return df, hourly, pivot, ratios


df, hourly, pivot, ratios = run_pipeline()
pivot

status,SUCCESS,REFUND,CANCELLED
item,,,
Cappuccino,1597.57,317.14,210.55
Croissant,1404.50,438.11,298.86
Espresso,814.22,253.54,237.92
Latte,1382.74,555.80,477.76
Muffin,942.99,254.50,214.26
Sandwich,2289.18,585.55,834.07


In [8]:
ratios.round(3)

,sales_count,refund_count,refund_ratio
item,,,
Cappuccino,93,15,0.161
Croissant,88,24,0.273
Espresso,78,21,0.269
Latte,74,22,0.297
Muffin,65,15,0.231
Sandwich,80,20,0.250


### Diskusjon: Task 5

Arbeidsflyten følger de sju stegene i oppgaven og lagrer fire filer i mappen `output/`. Outliers fjernes før NOK-korreksjonen, slik at 999.00 NOK ikke blir 9990.00. Prisene er i ulike valutaer og summeres uten omregning.

## Bruk av GenAI: oppsummering av chat-logg

**Emne:** INF201, Deliverable 2 (salgspipeline for Campus Coffee)
**Verktøy:** Claude (Anthropic)

Vi har jobbet sammen med det meste av prosjektet selv. Claude ble brukt som veiledning og støtte underveis, hovedsakelig til Task 4 og 5 og samlingen av notebooken.

### Hva vi ba om veiledning til
1. Hvordan kan vi gå frem for å bygge Task 4 med Pandas: DataFrame med tid som indeks, gruppering per time, pivot-tabell og refund ratio per item?
2. Hvordan kan vi sette funksjonene sammen til en arbeidsflyt i Task 5, og i hvilken rekkefølge bør rensingen skje (outliers før NOK-korreksjon)?
3. Hvordan får vi funksjonene fra Task 1-3 til å passe sammen med Task 4 og 5, for eksempel funksjonsnavn og hva funksjonene tar inn og returnerer?

### Hva GenAI ble brukt til
- **Forslag til kode:** Claude ga forslag til Task 4 og 5, som vi brukte som utgangspunkt og tilpasset funksjonene våre.
- **Tekst:** utkast til diskusjonene i markdown-cellene, som vi kortet ned og tilpasset.

### Hvordan resultatene ble kontrollert
- Vi testet funksjonene med små eksempler laget for hånd før vi brukte dem på de ekte dataene.
- Vi kjørte hele notebooken med Restart og Run All og sjekket at mappen `output/` fikk de fire filene.
- Vi leste gjennom funksjonene for å forstå hva de gjør.

### Begrensninger
- Claude kunne ikke se maskinene våre, så vi kjørte og sjekket all kode selv.
- Dataene fra generatoren er tilfeldige, så tallene blir forskjellige fra kjøring til kjøring.